
# Download KSSL soil hydraulic parameters for MIN3P

This notebook:

1. queries the USDA-NRCS Kellogg Soil Survey Laboratory (KSSL) Lab Data Mart with `py-soildb`
2. downloads all candidate pedons for each target soil series
3. summarizes data completeness so candidate pedons can be inspected manually
4. allows selected pedon IDs to be entered later without changing the query workflow
5. assigns KSSL horizons to simplified **A**, **B**, and **C** MIN3P layers
6. exports van Genuchten–Mualem parameters and saturated hydraulic conductivity to csv

The KSSL query includes the physical properties, calculations, and Rosetta key tables. KSSL Rosetta values are horizon-level pedotransfer estimates, not direct measurements of hydraulic conductivity.

## Important modeling choice

The original horizon-level KSSL/Rosetta values are always retained. The A–B–C aggregation below is an approximation:

- $\theta_r$, $\theta_s$, and $n$ use thickness-weighted arithmetic means
- $\alpha$ uses the geometric mean
- vertical $K_\mathrm{sat}$ uses the geometric mean
- $m = 1 - 1/n$

For final simulations, inspect both the original horizons and the aggregated layers.


In [ ]:
from __future__ import annotations

from pathlib import Path
import asyncio

import s3fs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from soildb import fetch_ldm

# Define a cache directory to cache KSSL queries
cache_dir = Path("./.cache")
if not cache_dir.exists():
    cache_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Get list of target soil series
s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

# Load in csv of site locations/names
site_df = pd.read_csv(f'{s3_base_path}/sites/site-locations.csv')

# Get list of target soil series
# (Replace "_" to convert Houston_Black to Houston Black)

target_series = site_df['site_id'].to_list()

## 1. Query KSSL by correlated series name

`py-soildb` performs a two-stage Lab Data Mart query: it first identifies matching pedons and then retrieves horizon-level laboratory data. We use a case-insensitive `corr_name` filter and request only standard soil horizons.

In [ ]:
redownload = False

async def fetch_series(series_name, overwrite_cached=False, cache_dir='./.cache'):
    """Fetch lab data mart table for all pedons matching a correlated taxon name

    Parameters:
    -----------
    series_name: str
        Name of series to fetch
    overwrite_cached: bool
        If true, overwrite cached file and query the series again (default: False)
    cache_dir: str
        Directory to use for cached data (default: ./.cache)

    Returns:
    --------
    df: pandas.DataFrame
        Series data from mart table"""

    # Check if this data has already been downloaded
    cache_file = f"{series}_kssl.parquet"
    cache_path = cache_dir / cache_file

    if cache_path.exists() and not overwrite_cached:
        return pd.read_parquet(cache_path)

    # Fetch Lab Data Mart table
    if series_name == 'HoustonBlack':
        search_name = 'Houston Black'
    else:
        search_name = series_name
    response = await fetch_ldm(x=search_name, what="samp_name",)

    df = response.to_pandas().copy()

    # Drop reporting layers because they aren't true horizons
    report_mask = df['layer_type'] == 'reporting layer'
    if report_mask.sum() > 0:
        df.drop(df.index[report_mask], inplace=True)

    # Save to cache folder
    df.to_parquet(cache_path, index=False)

    return df

series_frames = []

for series in target_series:
    print(f"Querying {series}...")

    frame = await fetch_series(series, cache_dir=cache_dir, overwrite_cached=redownload)
    # Move series_name to first column
    frame = frame.copy()
    frame['series_name'] = series
    frame = frame[['series_name'] + [c for c in frame.columns if c != 'series_name']]

    print(f"\t{len(frame)} horizon records")
    series_frames.append(frame)

all_horizons = pd.concat(series_frames, ignore_index=True, sort=False)

# Replace "Houston Black" with "Houston_Black"
hb_mask = all_horizons['series_name'] == "Houston Black"
all_horizons.loc[hb_mask, 'series_name'] = 'Houston_Black'

print(f"Total downloaded rows: {len(all_horizons):,}")
print(f"Total columns: {all_horizons.shape[1]}")

## 2. Check horizon and pedon coverage

In [ ]:
# Drop horizons without `hzn_bot`
# (These mostly occur at Cecil and don't have horizon designation either)
bot_mask = all_horizons['hzn_bot'].isna()
all_horizons = all_horizons[~bot_mask]

# Replace zero-length strings in 'hzn_master' with np.nan and drop them
# (Only one horizon in Cecil)
all_horizons['hzn_master'] = all_horizons['hzn_master'].replace('', np.nan)
all_horizons = all_horizons[~all_horizons['hzn_master'].isna()]

# Drop pedon 76290 from Cecil database, as it has ^A and ^C horizons
drop_mask = all_horizons['pedon_key'] == 76290
all_horizons = all_horizons[~drop_mask]

# Print all horizons that aren't A, B, C, O, E, or R
for i in all_horizons.index:
    hzn_label = all_horizons.loc[i, 'hzn_master']
    if hzn_label[0] not in ['A', 'B', 'C', 'O', 'E', 'R']:
        print(f"Unusual horizon label: {hzn_label} (series: {all_horizons.loc[i, 'series_name']}, pedon: {all_horizons.loc[i, 'pedon_key']})")

In [ ]:
# Print a summary of missing values in key columns
coverage_summary = (
    all_horizons
    .groupby('series_name', dropna=False)
    .agg(
        n_rows=('pedon_key', "size"),
        n_pedons=('pedon_key', "nunique"),
        missing_pedon_id=('pedon_key', lambda x: x.isna().sum()),
        missing_horizon_label=(
            'hzn_master',
            lambda x: x.isna().sum(),
        ),
        missing_top_depth=(
            'hzn_top',
            lambda x: x.isna().sum(),
        ),
        missing_bottom_depth=(
            'hzn_bot',
            lambda x: x.isna().sum(),
        ),
    )
    .reset_index()
)

display(coverage_summary)

## 3. Get depths/thicknesses of each horizon

First, check that they are increasing in the general pattern A --> B --> C

In [ ]:
# Based on the output below, create list of lab sample numbers to drop
drop_horizons = ['84P01611',  # Yolo A -> C -> A
                 '94P04864',  # Pullman, thin A horizon within otherwise thick B
                 '94P04847',  # Pullman, thin A horizon within otherwise thick B
                 '94P04627',  # Pullman, thin A horizon within otherwise thick B
                 '81P03891',  # Kuma, remove deep A horizon
                 '81P03892',  # Kuma, remove deep A horizon
                 '85P00967']  # Houston Black, overlapping B horizon
for horizon in drop_horizons:
    idx = all_horizons[all_horizons['labsampnum'] == horizon].index
    if len(idx) != 0:
        all_horizons.drop(idx, inplace=True)

change_horizons = {'89P04193': 'A',  # Kuma, BA in between A horizons
                   '95P03704': 'B',  # Pullman, deep AB horizon
                   '14N04890': 'B',  # Kalamazoo, deep E&B horizons
                   '14N04891': 'B',  # Kalamazoo, deep E&B horizons
                   '14N04892': 'B',  # Kalamazoo, deep E&B horizons
                   '14N04898': 'B',  # Kalamazoo, deep E&B horizons
                   '14N04899': 'B'}  # Kalamazoo, deep E&B horizons
for horizon, new_value in change_horizons.items():
    idx = all_horizons[all_horizons['labsampnum'] == horizon].index
    all_horizons.loc[idx, 'hzn_master'] = new_value

# Drop R horizons
rock_mask = all_horizons['hzn_master'] == "R"
all_horizons = all_horizons[~rock_mask]

# Simplify to primary horizon (e.g., "BC" --> "B" and "CB" --> "C")
all_horizons['hzn_master'] = all_horizons['hzn_master'].str[0]
all_horizons['hzn_master'] = all_horizons['hzn_master'].replace({"O": "A"})
all_horizons['hzn_master'] = all_horizons['hzn_master'].replace({"E": "A"})

series_depths = {}

for series, sdf in all_horizons.groupby('series_name'):
    rows = []

    for pedon, pdf in sdf.groupby('pedon_key'):
        pdf = pdf.sort_values(['hzn_top', 'hzn_bot']).copy()
        pdf = pdf[pdf['hzn_master'].isin(["A", "B", "C"])]

        # Collapse consecutive repeats, then check that horizons progress A -> B -> C.
        sequence = pdf['hzn_master'].loc[
            pdf['hzn_master'].ne(pdf['hzn_master'].shift())
        ].tolist()

        if sequence != sorted(sequence, key={"A": 0, "B": 1, "C": 2}.get):
            print(f"{series}, pedon {pedon}: horizons out of order "
                  f"({' -> '.join(sequence)})")

        row = {'pedon_key': pedon}
        for horizon in ["A", "B", "C"]:
            h = pdf[pdf['hzn_master'].eq(horizon)]
            row[f"{horizon}_top_cm"] = h['hzn_top'].min() if not h.empty else float("nan")
            row[f"{horizon}_bottom_cm"] = h['hzn_bot'].max() if not h.empty else float("nan")

        rows.append(row)

    series_depths[series] = pd.DataFrame(rows).set_index('pedon_key')

### Calculate mean A-->B and B-->C boundaries for each series

In [ ]:
series_names = list(series_depths)
index = pd.MultiIndex.from_product([series_names, ["A", "B", "C"]], names=["soil_series", "layer"])

soil_params = pd.DataFrame(index=index, columns=["top", "bottom", "theta_r", "theta_s", "alpha", "npar", "Ks", "bulk_density_third_bar"], dtype=float)

def boundary_mean(df, upper, lower):
    """Mean interface depth using both sides of a horizon boundary."""
    values = pd.concat([df[f"{upper}_bottom_cm"], df[f"{lower}_top_cm"]])

    return values.mean()

for series, df in series_depths.items():
    a_top = df["A_top_cm"].mean()

    if series == "Yolo":
        # No B horizon: A transitions directly to C.
        ac = boundary_mean(df, "A", "C")
        c_bottom = df["C_bottom_cm"].mean()

        soil_params.loc[(series, "A"), ["top", "bottom"]] = [0., ac]
        soil_params.loc[(series, "C"), ["top", "bottom"]] = [ac, 400.]

    elif series == "Pullman":
        # No C horizon: B extends to the mean observed B bottom.
        ab = boundary_mean(df, "A", "B")
        b_bottom = df["B_bottom_cm"].mean()

        soil_params.loc[(series, "A"), ["top", "bottom"]] = [0., ab]
        soil_params.loc[(series, "B"), ["top", "bottom"]] = [ab, 400.]

    else:
        ab = boundary_mean(df, "A", "B")
        bc = boundary_mean(df, "B", "C")
        c_bottom = df["C_bottom_cm"].mean()

        soil_params.loc[(series, "A"), ["top", "bottom"]] = [0., ab]
        soil_params.loc[(series, "B"), ["top", "bottom"]] = [ab, bc]
        soil_params.loc[(series, "C"), ["top", "bottom"]] = [bc, 400.]

soil_params

### Plot mean horizon depths for each series

In [ ]:
from matplotlib.patches import Rectangle

horizons = ["A", "B", "C"]
offsets = {"A": -0.28, "B": 0.00, "C": 0.28}
width = 0.24

fig, ax = plt.subplots(figsize=(10, 5))
horizon_colors = {"A": "#7A5C47", "B": "#B97A57", "C": "#CDBFAE"}

for i, (series, df) in enumerate(series_depths.items()):
    for horizon in horizons:
        top = df[f"{horizon}_top_cm"].dropna()
        bottom = df[f"{horizon}_bottom_cm"].dropna()

        if top.empty or bottom.empty:
            continue

        mean_top = top.mean()
        mean_bottom = bottom.mean()
        x = i + offsets[horizon]

        # Mean horizon interval
        ax.add_patch(Rectangle((x - width / 2, mean_top), width, mean_bottom - mean_top, alpha=0.25, color=horizon_colors[horizon]))

        # Mean top and bottom with +/- 1 SD
        ax.errorbar([x, x], [mean_top, mean_bottom], yerr=[top.std(), bottom.std()], fmt="o", capsize=4, linewidth=1.2, color=horizon_colors[horizon])

        # Horizon label inside rectangle
        ax.text(x, (mean_top + mean_bottom) / 2, horizon, ha="center", va="center", fontweight="bold")

ax.set(xlim=(-0.6, len(series_depths) - 0.4), ylabel="Depth (cm)", title="A, B, and C horizon depths across soil series",
       xticks=range(len(series_depths)), xticklabels=series_depths.keys())
ax.invert_yaxis()

fig.savefig('plots/MeanABChorizonDepths.png', dpi=300, bbox_inches='tight')

## 4. Add hydraulic function parameters to `soil_params`

In [ ]:
# First, use Rosetta to calculate van Genuchten–Mualem parameters for each horizon that is missing values
# By default, KSSL only calculates parameters if a sample has all six measured values (sand, silt, clay, clod bulk density, water content 33 kpa, water content 1500 kpa)
from rosetta import rosetta

print('NaN values by column:')
for param in ['sand_total', 'silt_total', 'clay_total', 'Ks', 'theta_r', 'theta_s', 'alpha', 'npar', 'bulk_density_third_bar', 'water_retention_third_bar', 'water_retention_15_bar']:
    print(f"    {param}: {all_horizons[param].isna().sum()}")

# Add ROSETTA_method column to later track how parameter values were derived
# Note that ROSETTA_method == 0 means values were derived from KSSL
new_col = pd.DataFrame(index=all_horizons.index, columns=['ROSETTA_method'], dtype=int)
all_horizons_imputed = pd.concat([all_horizons, new_col], axis=1)

ks_nan_mask = all_horizons['Ks'].isna()
ks_nan_idx = all_horizons_imputed.index[ks_nan_mask]
all_horizons_imputed.loc[~ks_nan_mask, 'ROSETTA_method'] = 0  # Method==0 means values calculated by KSSL

# Required order for ROSETTA:
# [sa (%), si (%), cl (%), bd (g/cm3), th33, th1500]
rosetta_cols = ['sand_total', 'silt_total', 'clay_total', 'bulk_density_third_bar', 'water_retention_third_bar', 'water_retention_15_bar']
data = all_horizons_imputed.loc[ks_nan_mask, rosetta_cols].values
means, stdevs, codes = rosetta(3, data, estimate_type='log')

# Means contains data in the following order
# theta_r, theta_s, alpha (1/cm), n, ksat (cm/d), k0 (cm/d), L (cm/d)
shfp_cols = ['theta_r', 'theta_s', 'alpha', 'npar', 'Ks']
all_horizons_imputed.loc[ks_nan_mask, shfp_cols] = means[:, :5]
all_horizons_imputed.loc[ks_nan_mask, 'ROSETTA_method'] = codes

print('\nAfter ROSETTA, NaN values by column:')
for param in shfp_cols:
    print(f"    {param}: {all_horizons_imputed[param].isna().sum()}")

In [ ]:
# Next, run several checks to make sure the parameter values are reasonable
qaqc_checks = {
    "theta_r": [
        ("outside [0, 1]", lambda s: ~s.between(0, 1)),
        ("greater than or equal to theta_s",
         lambda s: s >= all_horizons_imputed["theta_s"]),],
    "theta_s": [
        ("outside [0, 1]", lambda s: ~s.between(0, 1)),
        ("less than or equal to theta_r",
         lambda s: s <= all_horizons_imputed["theta_r"]),
    ],
    "alpha": [
        ("unusually large (> 10)", lambda s: (10**s) > 10),
    ],
    "npar": [
        ("less than or equal to 1", lambda s: (10**s) <= 1),
        ("unusually large (> 5)", lambda s: (10**s) > 5),
    ],
    "Ks": [
        ("unusually large (> 10,000)", lambda s: (10**s) > 10_000),
    ],
    "bulk_density_third_bar": [
        ("less than 0.9", lambda s: s < 0.9),
        ("greater than 2", lambda s: s > 2),
    ],
}
found_error = False
for parameter, checks in qaqc_checks.items():
    values = pd.to_numeric(all_horizons_imputed[parameter], errors="coerce")

    for description, check in checks:
        mask = values.notna() & check(values)
        if mask.sum() > 0:
            found_error = True
            print(f"\n{'=' * 60}\n{parameter}\n{'=' * 60}")
            print(f"{description}: {mask.sum()} value(s)")

            if mask.any():
                display(all_horizons_imputed.loc[mask, ['series_name', 'pedon_key', parameter]])
if not found_error:
    print('No odd parameter values were found.')

In [ ]:
log10_parameters = {"alpha", "npar", "Ks"}
params = [c for c in soil_params.columns if c not in ['top', 'bottom']]

for series in target_series:
    for horizon in ["A", "B", "C"]:
        # Yolo doesn't have B, Pullman doesn't have C
        if series == 'Yolo' and horizon == 'B':
            continue
        elif series == 'Pullman' and horizon == 'C':
            continue

        subset_mask = (all_horizons_imputed['series_name'] == series) & (all_horizons_imputed['hzn_master'] == horizon)
        subset = all_horizons_imputed.loc[subset_mask]

        for param in params:
            values = subset[param].dropna()

            if values.empty:
                if param != 'bulk_density_third_bar':
                    print(f'{series} {horizon} horizon: No {param} values.')
                    mean_value = np.nan
                else:
                    alt_col = 'bulk_density_third_bar_ws'
                    values = subset[alt_col].dropna()
                    if values.empty:
                        print(f'{series} {horizon} horizon: No {param} values.')
                        mean_value = np.nan
                    mean_value = values.mean()

            elif param in log10_parameters:
                # Values are stored as log10(parameter), so this is
                # the geometric mean in linear parameter space.
                mean_value = 10 ** values.mean()
            else:
                mean_value = values.mean()

            soil_params.loc[(series, horizon), param] = mean_value

In [ ]:
# Create a copy to output, renaming columns and converting units
output = soil_params.copy()

# Convert from cm (or cm/d) to m (or m/d)
for col in ['top', 'bottom', 'Ks']:
    output[col] /= 100
output['alpha'] *= 100  # Convert from 1/cm to 1/m

# Rename columns with units
rename_cols = {'top': 'top_m',
               'bottom': 'bottom_m',
               'alpha': 'alpha_m',
               'Ks': 'Ks_m.d',
               'npar': 'n',
               'bulk_density_third_bar': 'rho_g.cm3'}
output.rename(columns=rename_cols, inplace=True)

# Save to file
output.to_parquet(f'{s3_base_path}/input-data/processed-data/soil_physical_parameters.parquet', index=True)
output

## Box plots of soil hydraulic function parameters

In [ ]:
params_to_plot = [c for c in output.columns if c not in ['top_m', 'bottom_m']]
fig, axes = plt.subplots(1, len(params_to_plot), figsize=(3.2 * len(params_to_plot), 5),
                         tight_layout=True)

for ax, param in zip(axes, params_to_plot):
    data = output[param].dropna()

    ax.boxplot(data, showmeans=True, meanline=True)

    ax.set(ylabel=param, xticks=[])

    if 'Ks' in param:
        ax.set_yscale("log")

fig.suptitle("Parameter values across all soils and horizons")
fig.savefig('plots/SoilPhysicalParameterValues.png', dpi=300, bbox_inches='tight')